# Week 2, day 5 (morning) — Extra practice 12 SOLUTIONS: lambda, map and filter   (L06)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 12 — Lambda, map and filter. Run this once.
prices = [4.5, 12.0, 0.0, 7.25, 30.0]
quantities = [2, 1, 5, 3, 1]

books = [
    {"title": "Dune", "author": "Herbert", "year": 1965, "pages": 412},
    {"title": "Neuromancer", "author": "Gibson", "year": 1984, "pages": 271},
    {"title": "Ancillary Justice", "author": "Leckie", "year": 2013, "pages": 386},
    {"title": "Count Zero", "author": "Gibson", "year": 1986, "pages": 246},
]

print(len(books), "books")

### Question 1

`map` and `filter`. -> `[5.3999999999999995, 14.399999999999999, 0.0, 8.7, 36.0] 5`, then `[12.0, 7.25, 30.0] 3`.

Five in, five out from `map`; five in, three out from `filter`. Worksheet
03 Q5's distinction, under different names.

`4.5 * 1.2` is `5.3999999999999995` rather than `5.4`. Neither number is
exactly representable in binary, so the product carries the error. Fine to
print with `f"{p:.2f}"`; not fine to compare with `==`.

In [ ]:
taxed = list(map(lambda p: p * 1.2, prices))
print(taxed, len(taxed))

pricey = list(filter(lambda p: p > 5, prices))
print(pricey, len(pricey))

### Question 2

`map` over two lists. -> `[9.0, 12.0, 0.0, 21.75, 30.0] 72.75`, then `[9.0, 12.0, 0.0, 21.75] 4 from 5 prices`.

`map` accepts as many iterables as the function has parameters, walking them
in step — so a two-argument lambda gets one item from each.

And it **stops at the shortest**, exactly like `zip`. One price had no
quantity, so its line total was never computed and never mentioned: five
prices in, four totals out, `30.0` of revenue missing from a sum that still
looks like a sum.

Extra practice 02 Q4 is the same silence with `zip`, and the same fix:
check the lengths before you pair two lists that came from different
places.

In [ ]:
line_totals = list(map(lambda p, q: p * q, prices, quantities))
print(line_totals, sum(line_totals))

short = list(map(lambda p, q: p * q, prices, quantities[:-1]))
print(short, len(short), "from", len(prices), "prices")

### Question 3

Sorting, including on two fields. -> by year `['Dune', 'Neuromancer', 'Count Zero', 'Ancillary Justice']`; by pages descending `['Dune', 'Ancillary Justice', 'Neuromancer', 'Count Zero']`; then Gibson 1984, Gibson 1986, Herbert 1965, Leckie 2013.

**A tuple key sorts by its first element, then by the second to break ties**
— so `(author, year)` groups the two Gibsons together and orders them by
year within the group. One `sorted` call, two levels of ordering, no
intermediate list.

That generalises: a three-element tuple gives three levels. For mixed
directions — author ascending, year descending — negate a numeric field
(`(b["author"], -b["year"])`) or sort twice, least significant first, and
lean on the sort being stable.

In [ ]:
print([b["title"] for b in sorted(books, key=lambda b: b["year"])])
print([b["title"] for b in sorted(books, key=lambda b: b["pages"], reverse=True)])

by_author_year = sorted(books, key=lambda b: (b["author"], b["year"]))
for b in by_author_year:
    print(b["author"], b["year"], b["title"])

### Question 4

`max`, `min` and a total. -> `longest:  Dune 412`, `shortest: Count Zero 246`, then `1315 328.75`.

`max(books, key=...)` returns the **record**, so `longest["title"]` works.
That is almost always what you want: `max(b["pages"] for b in books)` gives
you 412 and no way back to the book.

`sum(b["pages"] for b in books)` is a generator expression, not a list —
no brackets, nothing built in memory, values produced as `sum` asks for
them.

An average of 328.75 pages across **four** books. Extra practice 05 Q3 and
worksheet 14 Q10: quote the denominator or the mean means nothing.

In [ ]:
longest = max(books, key=lambda b: b["pages"])
shortest = min(books, key=lambda b: b["pages"])
print("longest: ", longest["title"], longest["pages"])
print("shortest:", shortest["title"], shortest["pages"])

total_pages = sum(b["pages"] for b in books)
print(total_pages, total_pages / len(books))

### Question 5

`any` and `all`. -> `True`, `True`, `False`, `True`.

One line each, one boolean each, no loop and no flag variable. This is the
shape for a validation check.

Both short-circuit: `any` stops at the first `True`, `all` at the first
`False`. On a generator expression that means it can stop reading early,
which is why there are no brackets inside the parentheses.

And remember the empty case: `any([])` is `False`, `all([])` is **`True`**.
"Every one of no books is over 200 pages" is vacuously true, and that is
how an `all()` gate lets an empty feed straight through.

In [ ]:
print(any(b["year"] < 1970 for b in books))
print(all(b["pages"] > 200 for b in books))
print(any(b["author"] == "Asimov" for b in books))
print(all("title" in b for b in books))

### Question 6

A dictionary of lambdas. -> `double -> 24`, `half -> 6.0`, `square -> 144`, then `square of 9 is 81`.

Python has no `switch`, and a dict of functions is the usual replacement:
the key selects behaviour, and adding a fifth operation is one line rather
than another `elif`.

The values are function objects — **no parentheses** when you put them in,
parentheses when you take one out and call it. Extra practice 09 Q7 is the
same idea with `def`-defined functions, and that is usually the better
choice once the bodies grow past one expression.

`ops[chosen]` will raise `KeyError` for an unknown name. `ops.get(chosen)`
returns `None`, which then fails as "NoneType is not callable" somewhere
less helpful — so validate the key, do not soften the lookup.

In [ ]:
ops = {
    "double": lambda x: x * 2,
    "half": lambda x: x / 2,
    "square": lambda x: x ** 2,
}

for name, func in ops.items():
    print(name, "->", func(12))

chosen = "square"
print(chosen, "of 9 is", ops[chosen](9))

### Question 7

No lambda needed. -> `['Dune', 'Count Zero', 'Neuromancer', 'Ancillary Justice']` by length; `['Ancillary Justice', 'Count Zero', 'Dune', 'Neuromancer']`; `['a', 'b']`.

`key=len`, `key=str.lower`, `map(str.strip, ...)` — three useful operations,
no lambdas. If the lambda would only call one function on its argument,
pass that function.

`Count Zero` and `Neuromancer` are both 11 characters and kept their
original order, because Python's sort is stable. That guarantee is what
makes multi-pass sorting work.

The commented line is the trap: `sorted(books, key=len)` is valid and
sorts the dictionaries by **how many keys each has** — all four have 4, so
nothing moves. Code that runs, answers a question nobody asked, and
returns something that looks like a result.

In [ ]:
titles = [b["title"] for b in books]
print(sorted(titles, key=len))
print(sorted(titles, key=str.lower))
print(list(map(str.strip, [" a ", " b "])))

# sorted(books, key=len) would sort the DICTIONARIES by how many keys each
# has -- all four have 4, so it would return them unchanged. Valid code,
# meaningless question: len() of a dict is its key count.

### Question 8

`zip` is one-shot too. -> the five pairs, then **`[]`**, then `72.75`.

Same object, two `list()` calls, second one empty. `zip` is an iterator
like `map` and `filter`: it produces its pairs once, on demand, and is then
exhausted.

Worksheet 12 Q10 is the identical trap with `map`. It is worth listing the
family once: **`map`, `filter`, `zip`, `enumerate`, `reversed`, generator
expressions, and file objects** all behave this way. Anything you might
want to walk twice must be materialised with `list()` first.

The `sum` line works because it builds a fresh `zip`. That is the other
fix: rebuild rather than reuse.

In [ ]:
pairs = zip(prices, quantities)

print(list(pairs))
print(list(pairs))

print(sum(p * q for p, q in zip(prices, quantities)))

### Question 9

A lambda called with too few arguments. -> `3`, then `TypeError: <lambda>() missing 1 required positional argument: 'y'`.

A lambda is an ordinary function and checks its arguments the same way —
worksheet 09 Q11.

The name in the message is **`<lambda>`**, and that is the real cost of
anonymity. In a file with a dozen lambdas the traceback cannot tell you
which one; a `def`-defined function would have named itself. That is the
argument for `def` the moment a function is more than throwaway.

And note when it fires. `map(lambda p, q: p * q, prices)` raises the same
`TypeError`, but only when something **consumes** the map object — so the
traceback points at your `list()` call, several lines away from the
mistake. Laziness moves errors away from their cause.

In [ ]:
print((lambda x, y: x + y)(1, 2))

# This is SUPPOSED to raise: TypeError: <lambda>() missing 1 required
# positional argument: 'y'.
#
# A lambda is an ordinary function, so it checks its arguments the same way
# -- worksheet 09 Q11. Note the name in the message is `<lambda>`, which is
# the whole cost of anonymity: the traceback cannot tell you WHICH lambda.
#
# map(lambda p, q: p * q, prices) raises the same TypeError, but only when
# something consumes the map object -- the error appears at the list() call,
# not where the mistake is.
print((lambda x, y: x + y)(1))